# 合成CSVデータのスクリーニング可視化・件数確認ノート

このノートでは、以下を**探索用途**で確認します（学習は行いません）。

1. 指定ディレクトリ内の CTMC CSV を読み込み、スクリーニング前後の件数を確認する
2. `Q' (= Q_mle)` から抽出した `λ_i = Q'[i, i+1]` の分布を可視化し、スクリーニングの妥当性を点検する

- **入力**: `data_dir`（CSV格納ディレクトリ）
- **出力**: 保持/除外件数、除外理由集計、`λ` 可視化（N混在対応）

In [ ]:
# ==== 最初にここを編集 ====
# data_dir: CSV ファイルがあるディレクトリ
# recursive: サブディレクトリまで探索するか

data_dir = "./data"  # <- 最初にここを編集
recursive = True

# スクリーニング閾値
min_lambda = 1e-8
max_lambda = 1e6

# スクリーニング設定のON/OFF
check_nan_inf = True
require_structure = True
max_abs_diag_diff = 1e-8

# 表示設定
top_k_dropped_examples = 10

In [ ]:
from __future__ import annotations

import sys
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# ノートブックの実行場所に依らず import しやすくする
repo_root = Path.cwd()
src_dir = repo_root / "src"
if src_dir.exists() and str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from ctmc_surrogate.data.dataset_csv_loader import load_dir
from ctmc_surrogate.data.dataset_screening import (
    ScreeningConfig,
    extract_lambdas_from_Q,
    screen_datasets,
)

In [ ]:
# CSV を読み込む
try:
    datasets = load_dir(data_dir, recursive=recursive)
    print(f"ロード成功: {len(datasets)} 件")
    if len(datasets) == 0:
        print("CSV が 0 件です。data_dir / recursive の設定を見直してください。")
except Exception:
    print(f"読み込みに失敗しました: data_dir={data_dir}, recursive={recursive}")
    raise

In [ ]:
# スクリーニング実行
cfg = ScreeningConfig(
    min_lambda=min_lambda,
    max_lambda=max_lambda,
    check_nan_inf=check_nan_inf,
    require_structure=require_structure,
    max_abs_diag_diff=max_abs_diag_diff,
)

res = screen_datasets(datasets, cfg)

print(f"総件数: {len(datasets)}")
print(f"kept 件数: {len(res.kept)}")
print(f"dropped 件数: {len(res.dropped)}")

reason_counter = Counter(r.get("reason", "unknown") for r in res.report)
print("\n[dropped 理由別カウント]")
if reason_counter:
    for reason, cnt in reason_counter.most_common():
        print(f"- {reason}: {cnt}")
else:
    print("- dropped はありません")

print(f"\n[dropped 上位 {top_k_dropped_examples} 件]")
for i, item in enumerate(res.report[:top_k_dropped_examples], start=1):
    path = item.get("path", "(pathなし)")
    reason = item.get("reason", "(reasonなし)")
    idx = item.get("index")
    lam = item.get("lambda")
    detail = item.get("detail")
    base = f"{i:02d}. path={path}, reason={reason}"
    if idx is not None and lam is not None:
        base += f", index={idx}, lambda={lam:.6e}"
    if detail:
        base += f", detail={detail}"
    print(base)

In [ ]:
# λ 抽出関連ユーティリティ（N混在対応）
def group_by_n(datasets):
    grouped = defaultdict(list)
    for d in datasets:
        n = int(d.q_mle.shape[0])
        grouped[n].append(d)
    return dict(sorted(grouped.items(), key=lambda kv: kv[0]))


def extract_lambda_vectors(datasets):
    vectors = []
    errors = []
    for d in datasets:
        try:
            lambdas = extract_lambdas_from_Q(d.q_mle)
            vectors.append((d.path, lambdas))
        except Exception as e:
            errors.append((d.path, str(e)))
    return vectors, errors


kept_grouped = group_by_n(res.kept)
dropped_grouped = group_by_n(res.dropped)

print("[kept の N 内訳]")
if kept_grouped:
    for n, ds in kept_grouped.items():
        print(f"- N={n}: {len(ds)}")
else:
    print("- なし")

print("\n[dropped の N 内訳]")
if dropped_grouped:
    for n, ds in dropped_grouped.items():
        print(f"- N={n}: {len(ds)}")
else:
    print("- なし")

In [ ]:
# 可視化: N ごとに kept/dropped の λ を確認する

def report_by_path(report):
    return {r.get("path"): r for r in report}


report_map = report_by_path(res.report)
all_ns = sorted(set(kept_grouped.keys()) | set(dropped_grouped.keys()))

if not all_ns:
    print("可視化対象データがありません。")

for n in all_ns:
    kept_ds = kept_grouped.get(n, [])
    dropped_ds = dropped_grouped.get(n, [])

    kept_vectors, kept_errors = extract_lambda_vectors(kept_ds)
    dropped_vectors, dropped_errors = extract_lambda_vectors(dropped_ds)

    if kept_errors:
        print(f"[N={n}] kept λ抽出エラー")
        for p, msg in kept_errors:
            print(f"- path={p}: {msg}")

    if dropped_errors:
        print(f"[N={n}] dropped λ抽出エラー")
        for p, msg in dropped_errors:
            print(f"- path={p}: {msg}")

    k = max(n - 1, 0)
    x_positions = np.arange(k)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
    fig.suptitle(f"N={n} の λ 可視化")

    # (1) kept の箱ひげ図
    ax = axes[0]
    if kept_vectors and k > 0:
        mat = np.stack([v for _, v in kept_vectors], axis=0)  # (num_kept, n-1)
        per_index = [mat[:, i] for i in range(k)]
        ax.boxplot(per_index, positions=x_positions, widths=0.6)
        ax.set_title(f"kept λ 分布 (件数={len(kept_vectors)})")
    else:
        ax.set_title("kept λ 分布 (データなし)")

    ax.set_xlabel("i")
    ax.set_ylabel("λ_i")
    if k > 0:
        ax.set_xticks(x_positions)
    ax.axhline(min_lambda, linestyle="--")
    ax.axhline(max_lambda, linestyle="--")

    # (2) dropped の散布図（理由別）
    ax = axes[1]
    small_x, small_y = [], []
    large_x, large_y = [], []

    for path, vec in dropped_vectors:
        rec = report_map.get(path, {})
        reason = rec.get("reason")
        if reason not in {"lambda_too_small", "lambda_too_large"}:
            continue
        for i, lam in enumerate(vec):
            if reason == "lambda_too_small":
                small_x.append(i)
                small_y.append(lam)
            elif reason == "lambda_too_large":
                large_x.append(i)
                large_y.append(lam)

    if small_x:
        ax.scatter(small_x, small_y, label="lambda_too_small", alpha=0.7)
    if large_x:
        ax.scatter(large_x, large_y, label="lambda_too_large", alpha=0.7)

    if small_x or large_x:
        ax.legend()
        ax.set_title(f"dropped λ 散布 (閾値逸脱理由のみ, 件数={len(dropped_vectors)})")
    else:
        ax.set_title("dropped λ 散布 (閾値逸脱理由データなし)")

    ax.set_xlabel("i")
    ax.set_ylabel("λ_i")
    if k > 0:
        ax.set_xticks(x_positions)
    ax.axhline(min_lambda, linestyle="--")
    ax.axhline(max_lambda, linestyle="--")

    plt.show()

In [ ]:
# おまけ1: kept / dropped 全体ヒストグラム（Nごと）
for n in all_ns:
    kept_vectors, _ = extract_lambda_vectors(kept_grouped.get(n, []))
    dropped_vectors, _ = extract_lambda_vectors(dropped_grouped.get(n, []))

    kept_flat = np.concatenate([v for _, v in kept_vectors]) if kept_vectors else np.array([])
    dropped_flat = np.concatenate([v for _, v in dropped_vectors]) if dropped_vectors else np.array([])

    if kept_flat.size == 0 and dropped_flat.size == 0:
        continue

    plt.figure(figsize=(8, 4))
    if kept_flat.size > 0:
        plt.hist(kept_flat, bins=40, alpha=0.6, label="kept")
    if dropped_flat.size > 0:
        plt.hist(dropped_flat, bins=40, alpha=0.6, label="dropped")

    plt.axvline(min_lambda, linestyle="--")
    plt.axvline(max_lambda, linestyle="--")
    plt.xscale("log")
    plt.xlabel("λ (log scale)")
    plt.ylabel("count")
    plt.title(f"N={n} kept/dropped λ ヒストグラム")
    plt.legend()
    plt.show()

In [ ]:
# おまけ2: どの i が落ちやすいか（index 集計）
index_counter = Counter(
    int(r["index"]) for r in res.report if "index" in r and isinstance(r.get("index"), (int, np.integer))
)

print("[dropped index の頻度]")
if index_counter:
    for idx, cnt in index_counter.most_common():
        print(f"- i={idx}: {cnt}")
else:
    print("- index 情報のある dropped はありません")